# Module 01 — Image Fundamentals

> **Computer Vision Lab Track** | Foundational Module

This notebook is structured in four sections:

| Section | Theme |
|---------|-------|
| 1 | **The Working Blackbox** — run a pretrained model, see it work |
| 2 | **Anatomical Dissection** — open the image and inspect every layer |
| 3 | **The Sabotage Lab** — break it intentionally and understand why |
| 4 | **Clean Build + Feynman Test** — write reusable code and prove understanding |

---

In [ ]:
# ── Global Imports ────────────────────────────────────────────────────────────
import io
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
from PIL import Image

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

# Pretty-print helper
def banner(title, width=60):
    print('\n' + '─' * width)
    print(f'  {title}')
    print('─' * width)

print(f'torch      {torch.__version__}')
print(f'torchvision {torchvision.__version__}')
print(f'Device     : {"cuda" if torch.cuda.is_available() else "cpu"}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

---

## Section 1 — The Working Blackbox

We start with the finished product: a pretrained ResNet-18 correctly receiving a normalised image and producing sensible predictions. This gives us a **reference point** before we start dissecting and sabotaging.

Think of this as seeing the car drive before we open the hood.

In [ ]:
# ── 1A. Download a sample image ───────────────────────────────────────────────
IMAGE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg'

response = requests.get(IMAGE_URL, timeout=15)
response.raise_for_status()
pil_image = Image.open(io.BytesIO(response.content)).convert('RGB')

print(f'Downloaded image | size (W×H): {pil_image.size} | mode: {pil_image.mode}')

plt.figure(figsize=(5, 5))
plt.imshow(pil_image)
plt.title('Sample Image — Cute Dog', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 1B. Load class labels ─────────────────────────────────────────────────────
LABELS_URL = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
labels_resp = requests.get(LABELS_URL, timeout=15)
labels_resp.raise_for_status()
IMAGENET_CLASSES = [line.strip() for line in labels_resp.text.splitlines()]
print(f'Loaded {len(IMAGENET_CLASSES)} ImageNet class labels')
print('First 5 classes:', IMAGENET_CLASSES[:5])

In [ ]:
# ── 1C. Load pretrained ResNet-18 and run inference (CORRECTLY normalised) ────
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model = model.to(DEVICE)
model.eval()

# The standard ImageNet preprocessing pipeline
imagenet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),               # PIL uint8 [0,255] → float32 [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],      # ImageNet channel means (R, G, B)
        std=[0.229, 0.224, 0.225]        # ImageNet channel stds  (R, G, B)
    ),
])

input_tensor = imagenet_transform(pil_image).unsqueeze(0).to(DEVICE)  # [1, 3, 224, 224]

with torch.no_grad():
    logits = model(input_tensor)          # shape: [1, 1000]
    probs  = F.softmax(logits, dim=1)

top3_probs, top3_indices = torch.topk(probs, 3, dim=1)

banner('Top-3 Predictions (correctly normalised)')
for rank, (idx, prob) in enumerate(zip(top3_indices[0], top3_probs[0]), start=1):
    print(f'  #{rank}  {IMAGENET_CLASSES[idx.item()]:35s}  {prob.item()*100:6.2f}%')

---

## Section 2 — Anatomical Dissection

Now we crack open the image and look at every level of representation:
- PIL → NumPy → PyTorch tensor
- Individual colour channels visualised
- Pixel-value statistics
- The effect of normalisation step by step

In [ ]:
# ── 2A. PIL → NumPy ───────────────────────────────────────────────────────────
np_image = np.array(pil_image)

banner('PIL → NumPy')
print(f'  Type   : {type(np_image)}')
print(f'  Shape  : {np_image.shape}   (H, W, C)  — PIL/NumPy layout')
print(f'  Dtype  : {np_image.dtype}')
print(f'  Min    : {np_image.min()}')
print(f'  Max    : {np_image.max()}')
print(f'  Mean   : {np_image.mean():.2f}')
print(f'  Std    : {np_image.std():.2f}')

In [ ]:
# ── 2B. NumPy → PyTorch tensor (CHW, float32, [0, 1]) ────────────────────────
to_tensor = transforms.ToTensor()  # HWC uint8 → CHW float32, ÷255
tensor_01 = to_tensor(pil_image)   # shape: [3, H, W]

banner('ToTensor() → PyTorch [C, H, W]')
print(f'  Type   : {type(tensor_01)}')
print(f'  Shape  : {tensor_01.shape}   (C, H, W)  — PyTorch layout')
print(f'  Dtype  : {tensor_01.dtype}')
print(f'  Min    : {tensor_01.min().item():.4f}')
print(f'  Max    : {tensor_01.max().item():.4f}')
print(f'  Mean   : {tensor_01.mean().item():.4f}')
print(f'  Std    : {tensor_01.std().item():.4f}')

CHANNEL_NAMES = ['Red', 'Green', 'Blue']
for c, name in enumerate(CHANNEL_NAMES):
    ch = tensor_01[c]
    print(f'  Channel {name:5s} | min={ch.min().item():.3f}  max={ch.max().item():.3f}  '
          f'mean={ch.mean().item():.3f}  std={ch.std().item():.3f}')

In [ ]:
# ── 2C. Visualise individual R, G, B channels ─────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(pil_image)
axes[0].set_title('Original (RGB)', fontsize=12)
axes[0].axis('off')

channel_cmaps = ['Reds_r', 'Greens_r', 'Blues_r']
for c, (name, cmap) in enumerate(zip(CHANNEL_NAMES, channel_cmaps)):
    ch_np = tensor_01[c].numpy()   # [H, W], values in [0, 1]
    im = axes[c + 1].imshow(ch_np, cmap=cmap, vmin=0, vmax=1)
    axes[c + 1].set_title(f'{name} channel', fontsize=12)
    axes[c + 1].axis('off')
    plt.colorbar(im, ax=axes[c + 1], fraction=0.046, pad=0.04)

plt.suptitle('Individual RGB Channels (as grayscale intensity)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2D. Pixel-value histograms per channel (before normalisation) ─────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['salmon', 'mediumseagreen', 'cornflowerblue']

for c, (name, color) in enumerate(zip(CHANNEL_NAMES, colors)):
    pixel_vals = tensor_01[c].numpy().flatten()  # [0, 1]
    axes[c].hist(pixel_vals, bins=64, color=color, alpha=0.85, edgecolor='none')
    axes[c].set_title(f'{name} Channel — before normalisation', fontsize=11)
    axes[c].set_xlabel('Pixel value (÷255 → [0,1])', fontsize=9)
    axes[c].set_ylabel('Frequency', fontsize=9)
    axes[c].axvline(pixel_vals.mean(), color='black', linestyle='--', linewidth=1.5,
                    label=f'mean={pixel_vals.mean():.3f}')
    axes[c].legend(fontsize=9)

plt.suptitle('Pixel Value Distributions — [0, 1] (after ToTensor, before Normalize)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2E. Apply normalisation step by step and show distribution shift ──────────
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Step 1: raw uint8 from PIL (re-fetch as numpy for illustration)
raw_uint8 = torch.from_numpy(np_image).float()  # [H, W, C] — raw 0..255

# Step 2: ÷255
step2 = raw_uint8 / 255.0              # [H, W, C], values in [0, 1]
step2_chw = step2.permute(2, 0, 1)    # → [C, H, W]

# Step 3: subtract mean
step3 = step2_chw - IMAGENET_MEAN

# Step 4: divide by std
step4 = step3 / IMAGENET_STD          # final normalised tensor

banner('Normalisation step-by-step statistics (Red channel)')
for label, t in [
    ('Raw uint8 [0,255]',     raw_uint8[..., 0]),
    ('After ÷255  [0,1]',     step2_chw[0]),
    ('After − mean',           step3[0]),
    ('After ÷ std (final)',    step4[0]),
]:
    vals = t.numpy().flatten()
    print(f'  {label:28s}  min={vals.min():7.3f}  max={vals.max():7.3f}  '
          f'mean={vals.mean():7.3f}  std={vals.std():6.3f}')

In [ ]:
# ── 2F. Before / after histogram comparison ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for c, (name, color) in enumerate(zip(CHANNEL_NAMES, colors)):
    before = step2_chw[c].numpy().flatten()  # [0, 1]
    after  = step4[c].numpy().flatten()      # normalised

    ax_b = axes[0][c]
    ax_a = axes[1][c]

    ax_b.hist(before, bins=64, color=color, alpha=0.8, edgecolor='none')
    ax_b.axvline(before.mean(), color='black', linestyle='--', linewidth=1.5,
                 label=f'μ={before.mean():.3f}  σ={before.std():.3f}')
    ax_b.set_title(f'{name} — BEFORE (÷255 only)', fontsize=10)
    ax_b.set_xlabel('Pixel value', fontsize=9)
    ax_b.legend(fontsize=8)

    ax_a.hist(after, bins=64, color=color, alpha=0.8, edgecolor='none')
    ax_a.axvline(after.mean(), color='black', linestyle='--', linewidth=1.5,
                 label=f'μ={after.mean():.3f}  σ={after.std():.3f}')
    ax_a.set_title(f'{name} — AFTER ImageNet Normalize', fontsize=10)
    ax_a.set_xlabel('Pixel value', fontsize=9)
    ax_a.legend(fontsize=8)

plt.suptitle('Distribution Shift: Before vs After ImageNet Normalisation', fontsize=13)
plt.tight_layout()
plt.show()

---

## Section 3 — The Sabotage Lab

We deliberately break the preprocessing pipeline and observe the consequences.

**Hypothesis:** If we feed raw `[0, 255]` pixel values to a model trained on normalised `[−2.1, 2.6]` values, the model's activations will be wildly out of range and the predictions will be garbage.

In [ ]:
# ── 3A. Prepare ResNet-18 and the two input variants ─────────────────────────
# Resize & crop both versions to 224×224 (required input size)
resize_crop = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])
pil_224 = resize_crop(pil_image)

# Version A — RAW: convert to tensor WITHOUT normalisation (values 0–255)
raw_tensor = torch.from_numpy(np.array(pil_224)).float()  # [H, W, C], 0..255
raw_tensor = raw_tensor.permute(2, 0, 1).unsqueeze(0).to(DEVICE)  # [1, C, H, W]

# Version B — NORMALISED: proper ImageNet preprocessing
norm_tensor = imagenet_transform(pil_image).unsqueeze(0).to(DEVICE)  # [1, C, H, W]

banner('Input tensor statistics comparison')
print(f'  RAW   tensor shape: {raw_tensor.shape}  | range: [{raw_tensor.min().item():.1f}, {raw_tensor.max().item():.1f}]')
print(f'  NORM  tensor shape: {norm_tensor.shape}  | range: [{norm_tensor.min().item():.3f}, {norm_tensor.max().item():.3f}]')

In [ ]:
# ── 3B. Test 1 — Feed RAW pixels (0–255) — SABOTAGE ─────────────────────────
with torch.no_grad():
    logits_raw  = model(raw_tensor)
    probs_raw   = F.softmax(logits_raw, dim=1)

top3_probs_raw, top3_idx_raw = torch.topk(probs_raw, 3, dim=1)

banner('TEST 1 — RAW Input (0–255) — GARBAGE PREDICTIONS')
for rank, (idx, prob) in enumerate(zip(top3_idx_raw[0], top3_probs_raw[0]), start=1):
    print(f'  #{rank}  {IMAGENET_CLASSES[idx.item()]:40s}  {prob.item()*100:6.3f}%')
print()
print('  Note: The top-1 prediction has nearly no confidence.')
print('  Probability mass is spread across random, unrelated classes.')

In [ ]:
# ── 3C. Test 2 — Feed NORMALISED tensor — CORRECT ────────────────────────────
with torch.no_grad():
    logits_norm = model(norm_tensor)
    probs_norm  = F.softmax(logits_norm, dim=1)

top3_probs_norm, top3_idx_norm = torch.topk(probs_norm, 3, dim=1)

banner('TEST 2 — NORMALISED Input — CORRECT PREDICTIONS')
for rank, (idx, prob) in enumerate(zip(top3_idx_norm[0], top3_probs_norm[0]), start=1):
    print(f'  #{rank}  {IMAGENET_CLASSES[idx.item()]:40s}  {prob.item()*100:6.2f}%')

In [ ]:
# ── 3D. Statistical explanation — WHY does the raw input fail? ────────────────
banner('WHY IT FAILS — Statistical Analysis')

print('\n  What the model was trained to receive (ImageNet normalised statistics):')
print('  ┌────────────┬──────────┬──────────┐')
print('  │ Channel    │   Mean   │   Std    │')
print('  ├────────────┼──────────┼──────────┤')
for name, mu, sigma in zip(CHANNEL_NAMES,
                            [0.485, 0.456, 0.406],
                            [0.229, 0.224, 0.225]):
    print(f'  │ {name:<10} │  {mu:.3f}   │  {sigma:.3f}   │')
print('  └────────────┴──────────┴──────────┘')

print('\n  What the model actually received (raw 0–255 tensors):')
print('  ┌────────────┬──────────┬──────────┬────────────────────────────────┐')
print('  │ Channel    │   Mean   │   Std    │ Scale error (raw / normalised) │')
print('  ├────────────┼──────────┼──────────┼────────────────────────────────┤')

for c, (name, mu_exp, std_exp) in enumerate(zip(CHANNEL_NAMES,
                                                  [0.485, 0.456, 0.406],
                                                  [0.229, 0.224, 0.225])):
    # raw_tensor channel stats (float, 0–255 range)
    ch_raw = raw_tensor[0, c].cpu().numpy().flatten()
    mu_raw  = ch_raw.mean()
    std_raw = ch_raw.std()
    # What a 'typical' normalised value would be for this channel's mean pixel
    typical_norm = (mu_raw / 255.0 - mu_exp) / std_exp
    scale_error  = mu_raw / (typical_norm if typical_norm != 0 else 1.0)
    print(f'  │ {name:<10} │ {mu_raw:7.1f}  │ {std_raw:7.1f}  │  '
          f'{mu_raw:.1f} vs expected ≈{typical_norm:.2f}  ({abs(scale_error):.0f}×)  │')
print('  └────────────┴──────────┴──────────┴────────────────────────────────┘')

print('\n  Conclusion:')
print('  The raw pixel values are 100–600× larger than what the first')
print('  linear layer weights were calibrated for. This saturates')
print('  activations and invalidates every learned representation.')

In [ ]:
# ── 3E. Visual side-by-side: confidence distributions ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

probs_raw_np  = probs_raw[0].cpu().numpy()
probs_norm_np = probs_norm[0].cpu().numpy()

# Sort by probability for display
top20_idx_raw  = np.argsort(probs_raw_np)[::-1][:20]
top20_idx_norm = np.argsort(probs_norm_np)[::-1][:20]

axes[0].barh(range(20), probs_raw_np[top20_idx_raw][::-1], color='tomato', alpha=0.85)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels([IMAGENET_CLASSES[i][:25] for i in top20_idx_raw[::-1]], fontsize=7)
axes[0].set_title('RAW Input (0–255)\nTop-20 predictions', fontsize=11, color='tomato')
axes[0].set_xlabel('Probability', fontsize=9)

axes[1].barh(range(20), probs_norm_np[top20_idx_norm][::-1], color='steelblue', alpha=0.85)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([IMAGENET_CLASSES[i][:25] for i in top20_idx_norm[::-1]], fontsize=7)
axes[1].set_title('NORMALISED Input\nTop-20 predictions', fontsize=11, color='steelblue')
axes[1].set_xlabel('Probability', fontsize=9)

plt.suptitle('Sabotage Lab: Raw vs Normalised Predictions', fontsize=13)
plt.tight_layout()
plt.show()

---

## Section 4 — Clean Build + Feynman Test

Write clean, reusable, well-documented functions — and answer the three Feynman questions.

In [ ]:
# ── 4A. Clean function: preprocess_image ─────────────────────────────────────

def preprocess_image(pil_image: Image.Image,
                     input_size: int = 224,
                     mean: list = None,
                     std: list = None) -> torch.Tensor:
    """
    Convert a PIL RGB image into a normalised PyTorch tensor ready for
    inference with an ImageNet-pretrained model.

    Pipeline:
        1. Resize shortest edge to (input_size * 256/224) ≈ 256 for input_size=224
        2. CenterCrop to (input_size × input_size)
        3. ToTensor: PIL uint8 [0,255] → float32 [0.0, 1.0], layout [C,H,W]
        4. Normalize: (pixel - mean) / std  per channel
        5. Unsqueeze batch dimension: [C,H,W] → [1,C,H,W]

    Args:
        pil_image  : PIL.Image in 'RGB' mode.
        input_size : Spatial size expected by the model (default 224 for ResNets).
        mean       : Per-channel mean for normalisation. Defaults to ImageNet mean.
        std        : Per-channel std  for normalisation. Defaults to ImageNet std.

    Returns:
        torch.Tensor of shape [1, 3, input_size, input_size], dtype=float32.

    Example:
        >>> tensor = preprocess_image(pil_image)
        >>> tensor.shape
        torch.Size([1, 3, 224, 224])
    """
    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')

    mean = mean or [0.485, 0.456, 0.406]
    std  = std  or [0.229, 0.224, 0.225]

    resize_to = int(input_size * 256 / 224)  # maintain the 256→224 ratio

    pipeline = transforms.Compose([
        transforms.Resize(resize_to),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    return pipeline(pil_image).unsqueeze(0)  # [1, C, H, W]


# Smoke-test
t = preprocess_image(pil_image)
assert t.shape == torch.Size([1, 3, 224, 224]), 'Shape mismatch!'
assert t.dtype == torch.float32, 'Wrong dtype!'
print(f'preprocess_image: OK — shape={t.shape}, dtype={t.dtype}')
print(f'  value range: [{t.min().item():.3f}, {t.max().item():.3f}]')

In [ ]:
# ── 4B. Clean function: predict_top5 ─────────────────────────────────────────

def predict_top5(model: torch.nn.Module,
                 image_tensor: torch.Tensor,
                 class_labels: list,
                 device: torch.device = None) -> list:
    """
    Run inference on a single preprocessed image tensor and return the
    top-5 predicted class names with their probabilities.

    Args:
        model        : A torch.nn.Module (e.g. pretrained ResNet-18).
                       Will be automatically set to eval() mode.
        image_tensor : Tensor of shape [1, C, H, W], dtype=float32.
                       Must already be normalised.
        class_labels : List of class name strings, length == model output dim.
        device       : torch.device to run on. Defaults to CUDA if available.

    Returns:
        List of (class_name: str, probability: float) tuples, length 5,
        sorted by probability descending.

    Example:
        >>> results = predict_top5(model, tensor, IMAGENET_CLASSES)
        >>> for name, prob in results:
        ...     print(f'{name}: {prob*100:.2f}%')
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = model.to(device).eval()
    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        logits = model(image_tensor)          # [1, num_classes]
        probs  = F.softmax(logits, dim=1)     # [1, num_classes]

    top5_probs, top5_indices = torch.topk(probs, 5, dim=1)

    return [
        (class_labels[idx.item()], prob.item())
        for idx, prob in zip(top5_indices[0], top5_probs[0])
    ]


# Smoke-test
results = predict_top5(model, preprocess_image(pil_image), IMAGENET_CLASSES)
banner('predict_top5 smoke-test')
for rank, (name, prob) in enumerate(results, start=1):
    print(f'  #{rank}  {name:<35}  {prob*100:.2f}%')

In [ ]:
# ── 4C. Feynman Answers ───────────────────────────────────────────────────────

# =============================================================================
# FEYNMAN QUESTION 1:
# "Explain to a child: why can't we feed a raw 0-255 pixel image to a
#  pretrained neural network? What goes wrong and why?"
# =============================================================================

# FEYNMAN ANSWER 1:
#
# Imagine you trained a dog to catch balls by always throwing them at the same
# speed — say, 10 km/h. The dog learned exactly how far to run and when to
# jump. Now you throw the ball at 1000 km/h. The dog jumps too late and misses
# every time — not because it's a bad dog, but because the signal is so far
# outside what it was trained on.
#
# A pretrained neural network is the same. Every weight in every layer was
# optimised for inputs in the range ≈ [−2.1, 2.6] (ImageNet-normalised values).
# When we feed raw integers [0, 255], the first matrix multiplication produces
# activations that are ~100–600× too large. Batch normalisation layers, which
# were calibrated to see zero-mean, unit-variance activations, now see values
# with mean ~100 and std ~60. Every subsequent layer sees nonsense. The final
# softmax distributes probability across random classes with near-uniform
# confidence — the model is essentially guessing.

print('Feynman Answer 1 — annotated in code comments above.')
print()


# =============================================================================
# FEYNMAN QUESTION 2:
# "You have a cat image. When you apply RandomHorizontalFlip as augmentation,
#  the label stays 'cat'. But if you applied it to a licence plate OCR model,
#  would that be correct? Why or why not? What principle does this illustrate?"
# =============================================================================

# FEYNMAN ANSWER 2:
#
# For the cat classifier, a horizontally flipped cat is still a cat.
# The label is UNCHANGED by the transform → the transform is LABEL-INVARIANT
# for this task. Valid augmentation.
#
# For a licence plate OCR model, "ABC 123" horizontally flipped becomes
# "321 CBA" — a COMPLETELY DIFFERENT string. The label must change with the
# image, and in fact it changes in a way that produces an invalid plate.
# This transform DESTROYS the label → it is NOT label-invariant for OCR.
# Applying it naively would corrupt the training signal.
#
# The principle illustrated:
#   INVARIANCE — the label is unchanged by the transform. A transform is
#   a valid augmentation for a task only if it is label-invariant for that task.
#   (Contrast with EQUIVARIANCE — e.g. bounding boxes in detection,
#    where the label must transform TOGETHER WITH the image.)
#
# Always ask: "If I transform the image this way, does the correct label
# still hold?" If yes → valid. If no → do NOT use that augmentation.

print('Feynman Answer 2 — annotated in code comments above.')
print()


# =============================================================================
# FEYNMAN QUESTION 3:
# "A friend says 'I'll just divide all pixel values by 255 to normalise.'
#  Is this correct? What's missing, and how does proper ImageNet normalisation
#  differ?"
# =============================================================================

# FEYNMAN ANSWER 3:
#
# Dividing by 255 is a necessary first step but NOT sufficient on its own.
# It scales the range from [0, 255] to [0.0, 1.0], but the resulting
# distribution still has:
#   • Mean ≈ 0.45  (not 0)     → a DC bias that shifts every activation
#   • Std  ≈ 0.25  (not 1)     → spread different from training distribution
#
# Full ImageNet normalisation is a two-stage affine transform:
#
#   Step 1:  x = pixel / 255.0
#   Step 2:  x_norm = (x - mean_c) / std_c   for each channel c
#
#   Where:
#     mean = [0.485, 0.456, 0.406]   (dataset-wide per-channel mean)
#     std  = [0.229, 0.224, 0.225]   (dataset-wide per-channel std)
#
# This standardisation ensures x_norm has:
#   Mean ≈ 0.0  and  Std ≈ 1.0  per channel
#
# — exactly the distribution the model's weights were optimised for.
#
# Example (Red channel, pixel = 200):
#   ÷255 only  : 200/255 = 0.784
#   Properly   : (0.784 - 0.485) / 0.229 ≈ +1.31
#
# The ÷255-only value (0.784) differs from the expected value (+1.31) by 0.526,
# producing a systematic bias in every activation of the first layer.

# Demonstrate numerically:
example_pixel = 200.0
div255 = example_pixel / 255.0
properly_normalised = (div255 - 0.485) / 0.229

print('Feynman Answer 3 — Numerical demonstration:')
print(f'  Raw pixel value         : {example_pixel:.0f}')
print(f'  After ÷255 only         : {div255:.4f}   ← your friend stops here')
print(f'  After full normalisation: {properly_normalised:.4f}   ← what the model expects')
print(f'  Difference              : {abs(div255 - properly_normalised):.4f}  (systematic bias per pixel)')
print()
print('  Conclusion: ÷255 is necessary but NOT sufficient.')
print('  You must also subtract the channel mean and divide by channel std.')

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Tensor layout | PyTorch uses **[C, H, W]**; PIL/NumPy use **[H, W, C]** |
| Pixel ranges | Raw: `uint8 [0,255]` → `ToTensor()`: `float [0,1]` → Normalize: `float [−2, +2]` |
| Normalisation | **Required**. Two steps: ÷255, then `(x − μ) / σ` per channel |
| Invariance | Augmentation is valid only if it doesn't break the label |
| Equivariance | Geometric transforms must be applied to labels too |
| Dataset quality | Balance + diversity + no leakage = trustworthy metrics |

---
*Module 01 complete — proceed to **Module 02: Building a Custom Dataset & DataLoader***